# AAT6153 Chapter 2 — Hands-on 3
## Next-Word Prediction with a Feed-Forward Language Model
**Slides:** Text Prediction (37–43), Hands-on 3 (44–48)

A **language model** estimates P(next word | previous words). In this notebook you will:
1. Turn the complete works of Shakespeare into word ids (slide 44).
2. Create **(5-word window → next word)** training pairs with a sliding window: *self-supervised learning* (slides 39, 45).
3. Train an **Embedding → Flatten → Dense → softmax** model (slides 40, 46).
4. Suggest the **top-5 next words**, like a phone keyboard (slide 47).
5. Generate text with **temperature sampling** (slides 41–42, 48).

*Runtime: Runtime ▸ Change runtime type ▸ **T4 GPU** recommended (10 epochs).*

---
## Step 1: Load the Shakespeare corpus and convert it into word ids (slide 44)

In [ ]:
import numpy as np
import keras
from keras import layers

keras.utils.set_random_seed(7)
path = keras.utils.get_file("shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")
text = open(path, encoding="utf-8").read()
print(text[:250])

In [ ]:
vocab_size = 5000
vectorizer = layers.TextVectorization(max_tokens=vocab_size)
vectorizer.adapt(text.splitlines())
vocab = [str(w) for w in vectorizer.get_vocabulary()]
tokens = vectorizer([text])[0].numpy()    # the whole text as word ids
print(len(tokens), vocab[:10])

## Step 2: Create (input window, next word) pairs (slides 39, 45)

In [ ]:
window = 5                                 # use 5 words to predict the 6th
seqs = np.lib.stride_tricks.sliding_window_view(tokens, window + 1)
X, y = seqs[:, :-1], seqs[:, -1]
print(X.shape, y.shape)
print(" ".join(vocab[i] for i in X[100]), "->", vocab[y[100]])

## Step 3: Define and train the feed-forward language model (slides 40, 46)
The output layer has one neuron per vocabulary word (5,000) with softmax. Predicting the *exact* next word is hard, so we also track **top-5 accuracy**: how often the true word is among the 5 most likely suggestions.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(window,)),
    layers.Embedding(vocab_size, 64),          # (batch, 5, 64)
    layers.Flatten(),                          # (batch, 320)
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(vocab_size, activation="softmax")])  # P(next word)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy",
                       keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")])
model.summary()

In [ ]:
history = model.fit(X, y, epochs=10, batch_size=256, validation_split=0.1)

In [ ]:
import matplotlib.pyplot as plt
h = history.history
plt.plot(h["accuracy"], "b", label="train top-1"); plt.plot(h["val_accuracy"], "g", label="val top-1")
plt.plot(h["top5"], "b--", label="train top-5"); plt.plot(h["val_top5"], "g--", label="val top-5")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Next-word prediction"); plt.legend(); plt.show()

## Step 4: Suggest the top-5 next words (slide 47)

In [ ]:
def predict_next(seed, k=5):
    ids = vectorizer([seed])[0].numpy()[-window:]
    ids = np.pad(ids, (window - len(ids), 0))      # left-pad short seeds
    probs = model.predict(ids[None], verbose=0)[0]
    probs[:2] = 0                                  # skip padding and [UNK]
    top = np.argsort(probs)[::-1][:k]
    return [(vocab[i], round(float(probs[i]), 3)) for i in top]

print(predict_next("to be or not to"))
print(predict_next("my lord i will"))

---
## Temperature (slide 42)
Dividing the log-probabilities by **T** and re-normalising sharpens (T < 1) or flattens (T > 1) the distribution.

In [ ]:
def apply_temperature(probs, temperature):
    logits = np.log(probs + 1e-9) / temperature
    p = np.exp(logits - logits.max())
    return p / p.sum()

example = np.array([0.50, 0.25, 0.15, 0.07, 0.03])
labels = ["lord", "king", "good", "sir", "fair"]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, t in zip(axes, (0.2, 1.0, 1.5)):
    ax.bar(labels, apply_temperature(example, t)); ax.set_title(f"T = {t}")
axes[0].set_ylabel("Probability"); plt.show()

## Step 5: Generate text with temperature sampling (slide 48)

In [ ]:
def sample(probs, temperature=1.0):
    logits = np.log(probs + 1e-9) / temperature
    p = np.exp(logits - logits.max())
    p /= p.sum()
    return np.random.choice(len(p), p=p)

def generate(seed, n_words=20, temperature=0.8):
    words = seed.lower().split()
    for _ in range(n_words):
        ids = vectorizer([" ".join(words)])[0].numpy()[-window:]
        ids = np.pad(ids, (window - len(ids), 0))
        probs = model.predict(ids[None], verbose=0)[0]
        probs[:2] = 0                              # never pick padding / [UNK]
        words.append(vocab[sample(probs / probs.sum(), temperature)])
    return " ".join(words)

for t in (0.2, 0.8, 1.5):
    print(t, generate("the king is", temperature=t))

---
## Try it yourself
1. Which temperature reads best? Try your own seeds.
2. Change `window` to 3 and to 10. How do the parameter count (slide 40) and top-5 accuracy change?
3. Replace `Flatten()` with `GlobalAveragePooling1D()`. Why does accuracy drop? (Hint: word order, slide 43.)
4. Think about slide 43: what can this model *never* learn, however long it trains?

In [ ]:
# Use this cell for your experiments